# Validation of the $\Pi$-Transformer Model

In [ ]:
%load_ext autoreload

%autoreload 2

## 0 Imports and Predefinitions

In [ ]:
# %matplotlib widget
# import ipympl

import numpy as np
import os
import sys
import pandas as pd

# Set a fixed seed for reproducibility
np.random.seed(42)
if "torch" in sys.modules:
    import torch

    torch.manual_seed(42)

parent = os.path.abspath("/Users/maxikoehler/Documents/GitHub/diffpssi-ma-kohler/")
sys.path.insert(1, parent)
sys.path.append(
    str(os.path.dirname(os.path.dirname(os.path.abspath("ibb_trans_notebook.ipynb"))))
)

import matplotlib.pyplot as plt
import matplotlib as mpl
from diffpssi.tools import *

from diffpssi.power_sim_lib.simulator import PowerSystemSimulation as Pss
from diffpssi.power_sim_lib.simulator import Recorder
# from tools.calcs import *

parallel_sims = 1
time = np.arange(0, 5.005, 0.005)

In [ ]:
def load(S_n_trafo=4400, u_l=1.0, theta=0, x1=0.15):
    """
    Loads the grid data of the IBB transformer model.
    Returns: The grid data in form of a dictionary.

    """
    return {
        "base_mva": 2200,
        "f": 60,
        "slack_bus": "Bus 0",
        "base_voltage": 100,
        "busses": [
            ["name", "V_n"],
            ["Bus 0", 100],
            ["Bus 1", 100],
            ["Bus 2", 10],
        ],
        "lines": [
            ["name", "from_bus", "to_bus", "length", "unit", "R", "X", "B"],
            ["Line 1", "Bus 0", "Bus 1", 1, "p.u.", 0, 0.0484, 0],
        ],
        "transformers": [
            [
                "type",
                "name",
                "from_bus",
                "to_bus",
                "S_n",
                "V_n_from",
                "V_n_to",
                "R",
                "X",
                "u_l",
                "theta",
            ],
            ["simple", "T1", "Bus 1", "Bus 2", S_n_trafo, 100, 10, 0, x1, u_l, theta],
        ],
        "generators": {
            "GEN": [
                [
                    "name",
                    "bus",
                    "S_n",
                    "V_n",
                    "P",
                    "V",
                    "H",
                    "D",
                    "X_d",
                    "X_q",
                    "X_d_t",
                    "X_q_t",
                    "X_d_st",
                    "X_q_st",
                    "T_d0_t",
                    "T_q0_t",
                    "T_d0_st",
                    "T_q0_st",
                ],
                [
                    "G1",
                    "Bus 0",
                    11000,
                    100,
                    -1998,
                    0.995,
                    3.5e7,
                    0,
                    1.81,
                    1.76,
                    0.3,
                    0.65,
                    0.23,
                    0.23,
                    8.0,
                    1,
                    0.03,
                    0.07,
                ],
                [
                    "G2",
                    "Bus 2",
                    2200,
                    10,
                    1998,
                    1,
                    3.5,
                    0,
                    1.81,
                    1.76,
                    0.3,
                    0.65,
                    0.23,
                    0.23,
                    8.0,
                    1,
                    0.03,
                    0.07,
                ],
            ],
        },
    }

In [ ]:
def record_dict(simulation, call=False):
    record_dict = {
        "Bus 0: voltage magnitude": simulation.busses[0].get_value("voltage_mag"),
        "Bus 1: voltage magnitude": simulation.busses[1].get_value("voltage_mag"),
        "Bus 2: voltage magnitude": simulation.busses[2].get_value("voltage_mag"),
    }
    if call:
        return record_dict.values()
    else:
        return record_dict

## 1 Varying the Apparent Power $S_\mathrm{n}$

### 1.1 Sim Set-Up and Run

In [ ]:
S_n = [1100, 2200, 4400]
S_n_result = []

for S in S_n:
    sim = Pss(
        parallel_sims=parallel_sims,
        sim_time=5.005,
        time_step=0.005,
        solver="heun",
        grid_data=load(S_n_trafo=S),
    )

    sim.add_sc_event(1, 1.05, "Bus 0")
    rec = Recorder(sim=sim, recorder_dict=record_dict)
    sim.set_record_function(rec.record_fun)
    record_list = rec.record_list()
    t, recorder = sim.run()

    S_n_result.append(recorder)
    print(f"Current S_n: {sim.trafos[0].s_n}")

    del sim, t, recorder

In [ ]:
S_n_1100 = pd.DataFrame(S_n_result[0][0, :, :], columns=record_list)
S_n_1100["t"] = time
S_n_1100.set_index("t", inplace=True)

S_n_2200 = pd.DataFrame(S_n_result[1][0, :, :], columns=record_list)
S_n_2200["t"] = time
S_n_2200.set_index("t", inplace=True)

S_n_4400 = pd.DataFrame(S_n_result[2][0, :, :], columns=record_list)
S_n_4400["t"] = time
S_n_4400.set_index("t", inplace=True)

### 1.2 Inspection

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

for i, result in enumerate(S_n_result):
    for j in range(3):
        axs[i].plot(time, result[0, :, j], label=record_list[j])
    axs[i].set_title(f"Apparent Power: {S_n[i]} MVA", fontdict={"fontsize": 10})
    axs[i].grid()
    axs[i].legend()

fig.supxlabel("Time [s]")
fig.supylabel("Voltage magnitude [p.u.]")
fig.suptitle(r"Bus voltages for different transformer $S_\mathrm{n}$ ratings")

plt.tight_layout()
plt.show()

### 1.3 Loading PF Data

In [ ]:
data_pf_sn1100 = pd.read_csv(
    "./data/variation_sn/ibb_simple-transformer_valid_sn1100.csv", sep=";", dtype=float
)
data_pf_sn1100.rename(
    columns={
        "Time in s": "t",
        "Voltage, Magnitude in p.u.": "Bus 0: voltage magnitude",
        "Voltage, Magnitude in p.u..1": "Bus 1: voltage magnitude",
        "Voltage, Magnitude in p.u..2": "Bus 2: voltage magnitude",
    },
    inplace=True,
)
data_pf_sn1100.set_index("t", inplace=True)

data_pf_sn2200 = pd.read_csv(
    "./data/variation_sn/ibb_simple-transformer_valid_sn2200.csv", sep=";", dtype=float
)
data_pf_sn2200.rename(
    columns={
        "Time in s": "t",
        "Voltage, Magnitude in p.u.": "Bus 0: voltage magnitude",
        "Voltage, Magnitude in p.u..1": "Bus 1: voltage magnitude",
        "Voltage, Magnitude in p.u..2": "Bus 2: voltage magnitude",
    },
    inplace=True,
)
data_pf_sn2200.set_index("t", inplace=True)

data_pf_sn4400 = pd.read_csv(
    "./data/variation_sn/ibb_simple-transformer_valid_sn4400.csv", sep=";", dtype=float
)
data_pf_sn4400.rename(
    columns={
        "Time in s": "t",
        "Voltage, Magnitude in p.u.": "Bus 0: voltage magnitude",
        "Voltage, Magnitude in p.u..1": "Bus 1: voltage magnitude",
        "Voltage, Magnitude in p.u..2": "Bus 2: voltage magnitude",
    },
    inplace=True,
)
data_pf_sn4400.set_index("t", inplace=True)

pf_sn_result = [data_pf_sn1100, data_pf_sn2200, data_pf_sn4400]

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

for i, result in enumerate(pf_sn_result):
    for j in range(3):
        axs[i].plot(result.index.values, result.iloc[:, j], label=result.columns[j])
    axs[i].set_title(f"Apparent Power: {S_n[i]}", fontdict={"fontsize": 10})
    axs[i].grid()
    axs[i].legend()

fig.supxlabel("Time [s]")
fig.supylabel("Voltage magnitude [p.u.]")
fig.suptitle(
    r"Bus voltages for different transformer $S_\mathrm{n}$ ratings - PowerFactory"
)

plt.tight_layout()
plt.show()

### 1.4 Comparison

In [ ]:
# Calculate mean error for each S_n
S_n_errors = np.zeros((3, len(time), 3))
S_n_mean_errors = np.zeros(3)

S_n_error_lf = np.zeros((3, 3))

for i in range(3):
    ext_voltages = pf_sn_result[i].values
    model_voltage = S_n_result[i][0, :, :]
    errors = np.abs(ext_voltages - np.array(model_voltage))
    S_n_errors[i, :, :] = errors
    S_n_mean_errors[i] = np.mean(errors)

for i, error in enumerate(S_n_mean_errors):
    print(f"Mean error for S_n = {S_n[i]}: {np.round(error, 4)*100} %")

for i in range(3):
    for j in range(3):
        S_n_error_lf[i, j] = S_n_errors[i, 0, j]

    print(
        f"Initial value error for S_n = {S_n[i]}: {np.round(S_n_error_lf[i,:], 4)*100} %"
    )

## 2 Varying the Transformer Ratio $\underline{\vartheta}$

### 2.1 Sim Set-Up and Run

In [ ]:
theta = [0.9, 1.0, 1.1]
theta_result = []

for ratio in theta:
    sim = Pss(
        parallel_sims=parallel_sims,
        sim_time=5.005,
        time_step=0.005,
        solver="heun",
        grid_data=load(u_l=ratio),
    )

    sim.add_sc_event(1, 1.05, "Bus 0")
    rec = Recorder(sim=sim, recorder_dict=record_dict)
    sim.set_record_function(rec.record_fun)
    record_list = rec.record_list()
    t, recorder = sim.run()

    theta_result.append(recorder)
    print(f"Current ratio: {sim.trafos[0].u_l}")

    del sim, t, recorder

In [ ]:
theta_1100 = pd.DataFrame(theta_result[0][0, :, :], columns=record_list)
theta_1100["t"] = time
theta_1100.set_index("t", inplace=True)

theta_2200 = pd.DataFrame(theta_result[1][0, :, :], columns=record_list)
theta_2200["t"] = time
theta_2200.set_index("t", inplace=True)

theta_4400 = pd.DataFrame(theta_result[2][0, :, :], columns=record_list)
theta_4400["t"] = time
theta_4400.set_index("t", inplace=True)

### 2.2 Inspection

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

for i, result in enumerate(theta_result):
    for j in range(3):
        axs[i].plot(time, result[0, :, j], label=record_list[j])
    axs[i].set_title(f"Longitudinal Ratio: {theta[i]}", fontdict={"fontsize": 10})
    axs[i].grid()
    axs[i].legend()

fig.supxlabel("Time [s]")
fig.supylabel("Voltage magnitude [p.u.]")
fig.suptitle(r"Bus voltages for different transformer ratios")

plt.tight_layout()
plt.show()

### 2.3 Loading PF Data

In [ ]:
data_pf_theta09 = pd.read_csv(
    "./data/variation_theta/ibb_simple-transformer_valid_th09.csv", sep=";", dtype=float
)
data_pf_theta09.rename(
    columns={
        "Time in s": "t",
        "Voltage, Magnitude in p.u.": "Bus 0: voltage magnitude",
        "Voltage, Magnitude in p.u..1": "Bus 1: voltage magnitude",
        "Voltage, Magnitude in p.u..2": "Bus 2: voltage magnitude",
    },
    inplace=True,
)
data_pf_theta09.set_index("t", inplace=True)
data_pf_theta10 = pd.read_csv(
    "./data/variation_sn/ibb_simple-transformer_valid_sn4400.csv", sep=";", dtype=float
)
data_pf_theta10.rename(
    columns={
        "Time in s": "t",
        "Voltage, Magnitude in p.u.": "Bus 0: voltage magnitude",
        "Voltage, Magnitude in p.u..1": "Bus 1: voltage magnitude",
        "Voltage, Magnitude in p.u..2": "Bus 2: voltage magnitude",
    },
    inplace=True,
)
data_pf_theta10.set_index("t", inplace=True)

data_pf_theta11 = pd.read_csv(
    "./data/variation_theta/ibb_simple-transformer_valid_th11.csv", sep=";", dtype=float
)
data_pf_theta11.rename(
    columns={
        "Time in s": "t",
        "Voltage, Magnitude in p.u.": "Bus 0: voltage magnitude",
        "Voltage, Magnitude in p.u..1": "Bus 1: voltage magnitude",
        "Voltage, Magnitude in p.u..2": "Bus 2: voltage magnitude",
    },
    inplace=True,
)
data_pf_theta11.set_index("t", inplace=True)

pf_theta_result = [data_pf_theta09, data_pf_theta10, data_pf_theta11]

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

for i, result in enumerate(pf_theta_result):
    for j in range(3):
        axs[i].plot(result.index.values, result.iloc[:, j], label=result.columns[j])
    axs[i].set_title(f"Longitudinal Ratio: {theta[i]}", fontdict={"fontsize": 10})
    axs[i].grid()
    axs[i].legend()

fig.supxlabel("Time [s]")
fig.supylabel("Voltage magnitude [p.u.]")
fig.suptitle(
    r"Bus voltages for different transformer ratios - PowerFactory"
)

plt.tight_layout()
plt.show()

### 2.4 Comparison

In [ ]:
# Calculate mean error for each theta
theta_errors = np.zeros((3, len(time), 3))
theta_mean_errors = np.zeros(3)
theta_error_lf = np.zeros((3, 3))

for i in range(3):
    ext_voltages = pf_theta_result[i].values
    model_voltage = theta_result[i][0, :, :]
    errors = np.abs(ext_voltages - np.array(model_voltage))
    theta_errors[i, :, :] = errors
    theta_mean_errors[i] = np.mean(errors)

for i, error in enumerate(theta_mean_errors):
    print(f"Mean error for theta = {theta[i]}: {np.round(error, 4)*100} %")

for i in range(3):
    for j in range(3):
        theta_error_lf[i, j] = theta_errors[i, 0, j]

    print(
        f"Initial value error for theta = {theta[i]}: {np.round(theta_error_lf[i,:], 4)*100} %"
    )

## 3 Varying the Pos. Sequence Reactance $x_1$

### 3.1 Sim Set-Up and Run

In [ ]:
x1 = [0.15, 0.30, 0.60]
x1_result = []

for x in x1:
    sim = Pss(
        parallel_sims=parallel_sims,
        sim_time=5.005,
        time_step=0.005,
        solver="heun",
        grid_data=load(x1=x),
    )

    sim.add_sc_event(1, 1.05, "Bus 0")
    rec = Recorder(sim=sim, recorder_dict=record_dict)
    sim.set_record_function(rec.record_fun)
    record_list = rec.record_list()
    t, recorder = sim.run()

    x1_result.append(recorder)
    print(f"Current x1: {sim.trafos[0].x}")

    del sim, t, recorder

In [ ]:
x1_015 = pd.DataFrame(x1_result[0][0, :, :], columns=record_list)
x1_015["t"] = time
x1_015.set_index("t", inplace=True)

x1_030 = pd.DataFrame(x1_result[1][0, :, :], columns=record_list)
x1_030["t"] = time
x1_030.set_index("t", inplace=True)

x1_060 = pd.DataFrame(x1_result[2][0, :, :], columns=record_list)
x1_060["t"] = time
x1_060.set_index("t", inplace=True)

### 3.2 Inspection

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

for i, result in enumerate(x1_result):
    for j in range(3):
        axs[i].plot(time, result[0, :, j], label=record_list[j])
    axs[i].set_title(f"Transformer Reactance: {x1[i]} p.u.", fontdict={"fontsize": 10})
    axs[i].grid()
    axs[i].legend()

fig.supxlabel("Time [s]")
fig.supylabel("Voltage magnitude [p.u.]")
fig.suptitle(r"Bus voltages for different transformer reactances $x_1$")

plt.tight_layout()
plt.show()

### 3.3 Loading PF Data

In [ ]:
data_pf_x1_015 = pd.read_csv(
    "./data/variation_x1/ibb_simple-transformer_valid_x1015.csv", sep=";", dtype=float
)
data_pf_x1_015.rename(
    columns={
        "Time in s": "t",
        "Voltage, Magnitude in p.u.": "Bus 0: voltage magnitude",
        "Voltage, Magnitude in p.u..1": "Bus 1: voltage magnitude",
        "Voltage, Magnitude in p.u..2": "Bus 2: voltage magnitude",
    },
    inplace=True,
)
data_pf_x1_015.set_index("t", inplace=True)

data_pf_x1_030 = pd.read_csv(
    "./data/variation_x1/ibb_simple-transformer_valid_x1030.csv", sep=";", dtype=float
)
data_pf_x1_030.rename(
    columns={
        "Time in s": "t",
        "Voltage, Magnitude in p.u.": "Bus 0: voltage magnitude",
        "Voltage, Magnitude in p.u..1": "Bus 1: voltage magnitude",
        "Voltage, Magnitude in p.u..2": "Bus 2: voltage magnitude",
    },
    inplace=True,
)
data_pf_x1_030.set_index("t", inplace=True)

data_pf_x1_060 = pd.read_csv(
    "./data/variation_x1/ibb_simple-transformer_valid_x1060.csv", sep=";", dtype=float
)
data_pf_x1_060.rename(
    columns={
        "Time in s": "t",
        "Voltage, Magnitude in p.u.": "Bus 0: voltage magnitude",
        "Voltage, Magnitude in p.u..1": "Bus 1: voltage magnitude",
        "Voltage, Magnitude in p.u..2": "Bus 2: voltage magnitude",
    },
    inplace=True,
)
data_pf_x1_060.set_index("t", inplace=True)

pf_x1_result = [data_pf_x1_015, data_pf_x1_030, data_pf_x1_060]

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

for i, result in enumerate(pf_x1_result):
    for j in range(3):
        axs[i].plot(result.index.values, result.iloc[:, j], label=result.columns[j])
    axs[i].set_title(f"Transformer Reactance: {x1[i]} p.u.", fontdict={"fontsize": 10})
    axs[i].grid()
    axs[i].legend()

fig.supxlabel("Time [s]")
fig.supylabel("Voltage magnitude [p.u.]")
fig.suptitle(r"Bus voltages for different transformer reactances $x_1$ - PowerFactory")

plt.tight_layout()
plt.show()

### 3.4 Comparison

In [ ]:
# Calculate mean error for each theta
x1_errors = np.zeros((3, len(time), 3))
x1_mean_errors = np.zeros(3)
x1_error_lf = np.zeros((3, 3))

for i in range(3):
    ext_voltages = pf_x1_result[i].values
    model_voltage = x1_result[i][0, :, :]
    errors = np.abs(ext_voltages - np.array(model_voltage))
    x1_errors[i, :, :] = errors
    x1_mean_errors[i] = np.mean(errors)

for i, error in enumerate(x1_mean_errors):
    print(f"Mean error for x1 = {x1[i]}: {np.round(error, 4)*100} %")

for i in range(3):
    for j in range(3):
        x1_error_lf[i, j] = x1_errors[i, 0, j]

    print(
        f"Initial value error for x1 = {x1[i]}: {np.round(x1_error_lf[i,:], 4)*100} %"
    )

## 4 Varying the Phase Shifting Anlge $\Delta\phi$

### 4.1 Sim Set-Up and Run

In [ ]:
phi = [0, 5 * 30, 11 * 30]
phi_result = []

for angle in phi:
    sim = Pss(
        parallel_sims=parallel_sims,
        sim_time=5.005,
        time_step=0.005,
        solver="heun",
        grid_data=load(theta=angle),
    )

    sim.add_sc_event(1, 1.05, "Bus 0")
    rec = Recorder(sim=sim, recorder_dict=record_dict)
    sim.set_record_function(rec.record_fun)
    record_list = rec.record_list()
    t, recorder = sim.run()

    phi_result.append(recorder)
    print(f"Current angle: {sim.trafos[0].theta}")

    del sim, t, recorder

In [ ]:
phi_0 = pd.DataFrame(phi_result[0][0, :, :], columns=record_list)
phi_0["t"] = time
phi_0.set_index("t", inplace=True)

phi_5 = pd.DataFrame(phi_result[1][0, :, :], columns=record_list)
phi_5["t"] = time
phi_5.set_index("t", inplace=True)

phi_11 = pd.DataFrame(phi_result[2][0, :, :], columns=record_list)
phi_11["t"] = time
phi_11.set_index("t", inplace=True)

### 4.2 Inspection

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

for i, result in enumerate(phi_result):
    for j in range(3):
        axs[i].plot(time, result[0, :, j], label=record_list[j])
    axs[i].set_title(
        f"Transformer Vector Group Angle: {phi[i]}°", fontdict={"fontsize": 10}
    )
    axs[i].grid()
    axs[i].legend()

fig.supxlabel("Time [s]")
fig.supylabel("Voltage Magnitude [p.u.]")
fig.suptitle(r"Bus Voltages for different Transformer Vector Group Angles $\phi$")

plt.tight_layout()
plt.show()

### 4.3 Loading PF Data

In [ ]:
data_pf_phi0 = pd.read_csv(
    "./data/variation_sn/ibb_simple-transformer_valid_sn4400.csv", sep=";", dtype=float
)
data_pf_phi0.rename(
    columns={
        "Time in s": "t",
        "Voltage, Magnitude in p.u.": "Bus 0: voltage magnitude",
        "Voltage, Magnitude in p.u..1": "Bus 1: voltage magnitude",
        "Voltage, Magnitude in p.u..2": "Bus 2: voltage magnitude",
    },
    inplace=True,
)
data_pf_phi0.set_index("t", inplace=True)

data_pf_phi5 = pd.read_csv(
    "./data/variation_phi/ibb_simple-transformer_valid_vg5.csv", sep=";", dtype=float
)
data_pf_phi5.rename(
    columns={
        "Time in s": "t",
        "Voltage, Magnitude in p.u.": "Bus 0: voltage magnitude",
        "Voltage, Magnitude in p.u..1": "Bus 1: voltage magnitude",
        "Voltage, Magnitude in p.u..2": "Bus 2: voltage magnitude",
    },
    inplace=True,
)
data_pf_phi5.set_index("t", inplace=True)

data_pf_phi11 = pd.read_csv(
    "./data/variation_phi/ibb_simple-transformer_valid_vg11.csv", sep=";", dtype=float
)
data_pf_phi11.rename(
    columns={
        "Time in s": "t",
        "Voltage, Magnitude in p.u.": "Bus 0: voltage magnitude",
        "Voltage, Magnitude in p.u..1": "Bus 1: voltage magnitude",
        "Voltage, Magnitude in p.u..2": "Bus 2: voltage magnitude",
    },
    inplace=True,
)
data_pf_phi11.set_index("t", inplace=True)

pf_phi_result = [data_pf_phi0, data_pf_phi5, data_pf_phi11]

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

for i, result in enumerate(pf_phi_result):
    for j in range(3):
        axs[i].plot(result.index.values, result.iloc[:, j], label=result.columns[j])
    axs[i].set_title(
        f"Transformer Vector Group Angle: {phi[i]}°", fontdict={"fontsize": 10}
    )
    axs[i].grid()
    axs[i].legend()

fig.supxlabel("Time [s]")
fig.supylabel("Voltage Magnitude [p.u.]")
fig.suptitle(
    r"Bus Voltages for different Transformer Vector Group Angles $\phi$ - PowerFactory"
)

plt.tight_layout()
plt.show()

### 4.4 Comparison

In [ ]:
# Calculate mean error for each theta
phi_errors = np.zeros((3, len(time), 3))
phi_mean_errors = np.zeros(3)
phi_error_lf = np.zeros((3, 3))

for i in range(3):
    ext_voltages = pf_phi_result[i].values
    model_voltage = phi_result[i][0, :, :]
    errors = np.abs(ext_voltages - np.array(model_voltage))
    phi_errors[i, :, :] = errors
    phi_mean_errors[i] = np.mean(errors)

for i, error in enumerate(phi_mean_errors):
    print(f"Mean error for phi = {phi[i]}: {np.round(error, 4)*100} %")

for i in range(3):
    for j in range(3):
        phi_error_lf[i, j] = phi_errors[i, 0, j]

    print(
        f"Initial value error for phi = {phi[i]}: {np.round(phi_error_lf[i,:], 4)*100} %"
    )

## Comparative Plots for the Thesis

### Apparent Power Rating

In [ ]:
fig, axs = plt.subplots(3, 2, figsize=(10, 9), sharex=True)

for i, result in enumerate(S_n_result):
    for j in range(3):
        axs[i, 0].plot(time, result[0, :, j], label=record_list[j])
    axs[i, 0].set_title(f"Apparent Power: {S_n[i]} MVA", fontdict={"fontsize": 10})
    axs[i, 0].grid()
axs[2, 0].legend()

for i, result in enumerate(pf_sn_result):
    for j in range(3):
        axs[i, 1].plot(result.index.values, result.iloc[:, j], label=result.columns[j])
    axs[i, 1].set_title(f"Apparent Power: {S_n[i]} MVA", fontdict={"fontsize": 10})
    axs[i, 1].grid()
axs[2, 1].legend()

fig.supxlabel("Time in s")
fig.supylabel("Voltage Magnitude in p.u.")
# fig.suptitle(r'Bus voltages for different transformer $S_\mathrm{n}$ ratings - diffpssi vs. PowerFactory')

plt.savefig("./data/s_n_comp_complete.pdf")
# plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

for i in range(3):
    axs[i].plot(time, S_n_1100.iloc[:, i], label="diffpssi")
    axs[i].plot(
        time,
        data_pf_sn1100.iloc[:, i],
        label="PowerFactory",
        color=ees_red,
        linestyle="dashed",
    )
    axs[i].set_title(f"Bus number {i}", fontdict={"fontsize": 10})
    axs[i].grid()
    axs[i].legend()

fig.supxlabel("Time in s")
fig.supylabel("Voltage Magnitude in p.u.")
# fig.suptitle(r'Bus voltages for transformer rating 1100 MVA - diffpssi vs. PowerFactory')

plt.savefig("./data/s_n_comp_1100mva.pdf")
# plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 3))

plt.plot(time, S_n_1100.iloc[:, 2], label="diffpssi")
plt.plot(
    time,
    data_pf_sn1100.iloc[:, 2],
    label="PowerFactory",
    color=ees_red,
    linestyle="dashed",
)

plt.grid()
plt.legend()

plt.xlabel("Time in s")
plt.ylabel("Voltage Magnitude in p.u.")
# fig.suptitle(r'Bus voltages for transformer rating 1100 MVA - diffpssi vs. PowerFactory')

# plt.savefig('./data/s_n_comp_1100mva.pdf')
# plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

for i in range(3):
    for j in range(3):
        axs[i].plot(time, S_n_errors[i, :, j], label=f"Bus {j}: Voltage Error")
        # axs[i].plot(time, pf_sn_result[i].iloc[:, j], label=record_list[j])
    axs[i].set_title(f"Apparent Power: {S_n[i]} MVA", fontdict={"fontsize": 10})
    axs[i].grid()
axs[1].legend()

fig.supxlabel("Time in s")
fig.supylabel("Voltage Magnitude in p.u.")
# fig.suptitle(r'Bus voltages errors for different transformer $S_\mathrm{n}$ ratings - diffpssi vs. PowerFactory')

plt.tight_layout()

plt.savefig("./data/s_n_comp_errors.pdf")
plt.show()

In [ ]:
plt.figure(figsize=(8, 3))
for j in range(3):
    plt.plot(time, S_n_errors[0, :, j], label=f"Bus {j}: Voltage Error")

plt.grid()
plt.legend()

plt.xlabel("Time in s")
plt.ylabel("Voltage Magnitude in p.u.")
# fig.suptitle(r'Bus voltages for transformer rating 1100 MVA - diffpssi vs. PowerFactory')

# plt.savefig('./data/s_n_comp_1100mva.pdf')
# plt.tight_layout()
plt.show()

### Transformer Ratio

In [ ]:
fig, axs = plt.subplots(3, 2, figsize=(10, 9), sharex=True)

for i, result in enumerate(theta_result):
    for j in range(3):
        axs[i, 0].plot(time, result[0, :, j], label=record_list[j])
    axs[i, 0].set_title(
        f"Longitudinal Ratio: {theta[i]} p.u.", fontdict={"fontsize": 12}
    )
    axs[i, 0].grid()
axs[2, 0].legend()

for i, result in enumerate(pf_theta_result):
    for j in range(3):
        axs[i, 1].plot(result.index.values, result.iloc[:, j], label=result.columns[j])
    axs[i, 1].set_title(
        f"Longitudinal Ratio: {theta[i]} p.u.", fontdict={"fontsize": 12}
    )
    axs[i, 1].grid()
axs[2, 1].legend()

fig.supxlabel("Time in s")
fig.supylabel("Voltage Magnitude in p.u.")
# fig.suptitle(r'Bus voltages for different transformer ratios $\underline{\vartheta}$ - diffpssi vs. PowerFactory')

plt.savefig("./data/theta_comp_complete.pdf")
# plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

for i in range(3):
    axs[i].plot(time, theta_1100.iloc[:, i], label="diffpssi")
    axs[i].plot(
        time,
        data_pf_theta09.iloc[:, i],
        label="PowerFactory",
        color=ees_red,
        linestyle="dashed",
    )
    axs[i].set_title(f"Bus number {i}", fontdict={"fontsize": 12})
    axs[i].grid()
    axs[i].legend()

fig.supxlabel("Time in s")
fig.supylabel("Voltage Magnitude in p.u.")
# fig.suptitle(r'Bus voltages for transformer longitudinal ratio 0.9 p.u. - diffpssi vs. PowerFactory')

plt.savefig("./data/theta_comp_ratio09.pdf")
# plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 3))

plt.plot(time, theta_1100.iloc[:, 2], label="diffpssi")
plt.plot(
    time,
    data_pf_theta09.iloc[:, 2],
    label="PowerFactory",
    color=ees_red,
    linestyle="dashed",
)

plt.grid()
plt.legend()

plt.xlabel("Time in s")
plt.ylabel("Voltage Magnitude in p.u.")
# fig.suptitle(r'Bus voltages for transformer rating 1100 MVA - diffpssi vs. PowerFactory')

# plt.savefig('./data/s_n_comp_1100mva.pdf')
# plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

for i in range(3):
    for j in range(3):
        axs[i].plot(time, theta_errors[i, :, j], label=f"Bus {j}: Voltage Error")
        # axs[i].plot(time, pf_sn_result[i].iloc[:, j], label=record_list[j])
    axs[i].set_title(f"Logitudinal Ratio: {theta[i]} p.u.", fontdict={"fontsize": 12})
    axs[i].grid()
axs[1].legend()

fig.supxlabel("Time in s")
fig.supylabel("Voltage Magnitude in p.u.")
# fig.suptitle(r'Bus voltages errors for different transformer ratios $\underline{\vartheta}$ - diffpssi vs. PowerFactory')

plt.savefig("./data/theta_comp_errors.pdf")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 3))
for j in range(3):
    plt.plot(time, theta_errors[0, :, j], label=f"Bus {j}: Voltage Error")

plt.grid()
plt.legend()

plt.xlabel("Time in s")
plt.ylabel("Voltage Magnitude in p.u.")
# fig.suptitle(r'Bus voltages for transformer rating 1100 MVA - diffpssi vs. PowerFactory')

# plt.savefig('./data/s_n_comp_1100mva.pdf')
# plt.tight_layout()
plt.show()